![alt text](../../pythonexposed-high-resolution-logo-black.jpg "Optionele titel")

## Oefening 1: 
In data science gebruiken we vaak grote datasets.  Vaak worden die in batches ingelezen en gepresenteerd aan het model om te trainen - indien we alles in 1 keer zouden aanbieden zou onze PC crashen.  Daarom zijn iterators en generators handig: we kunnen de pipeline 'voeden' met een deel van de data, en dit verwerken.  

Stel je een dataset ('DATA/large_dataset.txt') voor met 20 lijntjes, die we in 4 'batches' van 5 lijen willen verwerken. 

Schrijf een Generator om batches in te lezen, en implementeer vervolgens ook een Iterator class met gelijkaardige functionaliteit.

Tip: voeg eens een lijntje toe aan large_dataset.txt.  Krijg je dit ook te zien?

In [40]:
# Oplossing met een generator

In [19]:
def read_file_in_batches(file_path, batch_size):    
    with open(file_path, 'r') as file:
        print("opening file")
        batch = []
        for line in file:
            print(f"Reading line: {line.strip()}", end = " - ")  # Debug: toont elke gelezen lijn
            batch.append(line.strip())
            if len(batch) == batch_size:
                print(f"\nYielding batch: {batch}\n")
                yield batch
                batch = []
        if batch: # indien de laatste batch minder is dan batch_size (aantal_regels %4 != 0)
            print(f"Yielding last batch: {batch}")
            yield batch
    print("Finished!")

In [20]:
batch_of_data = read_file_in_batches('DATA/large_dataset.txt', 5)

In [21]:
from time import sleep

for batch in batch_of_data:
    print(batch)
    sleep(1)

opening file
Reading line: Line 1 - Reading line: Line 2 - Reading line: Line 3 - Reading line: Line 4 - Reading line: Line 5 - 
Yielding batch: ['Line 1', 'Line 2', 'Line 3', 'Line 4', 'Line 5']

['Line 1', 'Line 2', 'Line 3', 'Line 4', 'Line 5']
Reading line: Line 6 - Reading line: Line 7 - Reading line: Line 8 - Reading line: Line 9 - Reading line: Line 10 - 
Yielding batch: ['Line 6', 'Line 7', 'Line 8', 'Line 9', 'Line 10']

['Line 6', 'Line 7', 'Line 8', 'Line 9', 'Line 10']
Reading line: Line 11 - Reading line: Line 12 - Reading line: Line 13 - Reading line: Line 14 - Reading line: Line 15 - 
Yielding batch: ['Line 11', 'Line 12', 'Line 13', 'Line 14', 'Line 15']

['Line 11', 'Line 12', 'Line 13', 'Line 14', 'Line 15']
Reading line: Line 16 - Reading line: Line 17 - Reading line: Line 18 - Reading line: Line 19 - Reading line: Line 20 - 
Yielding batch: ['Line 16', 'Line 17', 'Line 18', 'Line 19', 'Line 20']

['Line 16', 'Line 17', 'Line 18', 'Line 19', 'Line 20']
Reading line: 

In [51]:
# Handmatig itereren
batch_of_data = read_file_in_batches('DATA/large_dataset.txt', 5)
# Gebruik `next()` om batches handmatig op te halen
print(next(batch_of_data))  # Leest de eerste batch
print(next(batch_of_data))  # Leest de tweede batch
print(next(batch_of_data))  # Leest de derde batch
print(next(batch_of_data))  # Leest de vierde batch
print(next(batch_of_data))  # Leest de vijfde batch

Reading line: Line 1 - Reading line: Line 2 - Reading line: Line 3 - Reading line: Line 4 - Reading line: Line 5 - 
Yielding batch: ['Line 1', 'Line 2', 'Line 3', 'Line 4', 'Line 5']

['Line 1', 'Line 2', 'Line 3', 'Line 4', 'Line 5']
Reading line: Line 6 - Reading line: Line 7 - Reading line: Line 8 - Reading line: Line 9 - Reading line: Line 10 - 
Yielding batch: ['Line 6', 'Line 7', 'Line 8', 'Line 9', 'Line 10']

['Line 6', 'Line 7', 'Line 8', 'Line 9', 'Line 10']
Reading line: Line 11 - Reading line: Line 12 - Reading line: Line 13 - Reading line: Line 14 - Reading line: Line 15 - 
Yielding batch: ['Line 11', 'Line 12', 'Line 13', 'Line 14', 'Line 15']

['Line 11', 'Line 12', 'Line 13', 'Line 14', 'Line 15']
Reading line: Line 16 - Reading line: Line 17 - Reading line: Line 18 - Reading line: Line 19 - Reading line: Line 20 - 
Yielding batch: ['Line 16', 'Line 17', 'Line 18', 'Line 19', 'Line 20']

['Line 16', 'Line 17', 'Line 18', 'Line 19', 'Line 20']
Reading line: Line 21 - Yie

In [37]:
# Oplossing met een Iterator class

In [54]:
class FileBatchReader:
    def __init__(self, file_path, batch_size):
        """
        Initialiseer de FileBatchReader.

        Parameters:
        - file_path: Pad naar het bestand dat moet worden gelezen.
        - batch_size: Het aantal lijnen per batch.
        """
        self.file_path = file_path
        self.batch_size = batch_size
        self.file = None  # Bestand dat we openen
        self._open_file()  # Open bestand bij initialisatie

    def _open_file(self):
        """Open het bestand voor lezen."""
        self.file = open(self.file_path, 'r')

    def __iter__(self):
        """
        Maak de klasse iterabel door zichzelf als iterator terug te geven.
        """
        return self

    def __next__(self):
        """
        Retourneer de volgende batch.
        
        Raises:
        - StopIteration: Wanneer het einde van het bestand is bereikt.
        """
        batch = []
        for _ in range(self.batch_size):
            line = self.file.readline()
            if not line:  # Einde van bestand bereikt
                if batch:  # Retourneer resterende regels
                    return batch
                else:
                    self.close()  # Sluit bestand als we klaar zijn
                    raise StopIteration()
            batch.append(line.strip())
        return batch

    def close(self):
        """Sluit het bestand."""
        if self.file:
            self.file.close()
            self.file = None  # Zorg ervoor dat we weten dat het bestand gesloten is

    def __del__(self):
        """Zorg ervoor dat het bestand wordt gesloten bij het vernietigen van de instantie."""
        self.close()


In [56]:
file_reader = FileBatchReader('DATA/large_dataset.txt', 4)

for batch in file_reader:
    print(f"Processed batch: {batch}")

file_reader.close()  # Dit kun je expliciet doen, maar is niet nodig als je iterator klaar is.

Processed batch: ['Line 1', 'Line 2', 'Line 3', 'Line 4']
Processed batch: ['Line 5', 'Line 6', 'Line 7', 'Line 8']
Processed batch: ['Line 9', 'Line 10', 'Line 11', 'Line 12']
Processed batch: ['Line 13', 'Line 14', 'Line 15', 'Line 16']
Processed batch: ['Line 17', 'Line 18', 'Line 19', 'Line 20']
Processed batch: ['Line 21']


## Oefening 2: 
Genormaliseerde Batch Generator  

Maak een iterator-klasse genaamd BatchNormalizer die een dataset (een lijst van lijsten met numerieke waarden) in batches verdeelt en deze batch per batch normaliseert. Normalisatie houdt in dat je van elke waarde in de batch het minimum aftrekt en deelt door het bereik (max - min).  

Vereisten:  

- De iterator moet batches van een opgegeven grootte teruggeven.
- Indien de dataset niet exact kan worden verdeeld, bevat de laatste batch de resterende elementen.
- Normalisatie gebeurt per batch, niet over de hele dataset.

In [1]:
# Oplossing:

class BatchNormalizer:
    def __init__(self, data, batch_size):
        """
        Initialiseer de iterator met de dataset en batchgrootte.

        :param data: Een lijst van lijsten met numerieke waarden
        :param batch_size: De grootte van de batches
        """
        self.data = data
        self.batch_size = batch_size
        self.current_index = 0

    def __iter__(self):
        """Maak het object zelf iterable."""
        return self

    def __next__(self):
        """
        Retourneer de volgende genormaliseerde batch.

        :raises StopIteration: Als er geen batches meer zijn
        """
        if self.current_index >= len(self.data):
            raise StopIteration

        # Haal de huidige batch
        batch = self.data[self.current_index:self.current_index + self.batch_size]
        self.current_index += self.batch_size

        # Normaliseer de batch
        min_val = min(min(row) for row in batch)
        max_val = max(max(row) for row in batch)
        range_val = max_val - min_val

        if range_val == 0:  # Voorkom deling door nul; niks te normaliseren want alles gelijk!
            normalized_batch = [[0 for _ in row] for row in batch] # alle values op nul
        else:
            normalized_batch = [
                [(val - min_val) / range_val for val in row] for row in batch
            ]

        return normalized_batch

In [2]:
# Testvoorbeeld
data = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
    [10, 11, 12]
]
batch_size = 2Methode
normalizer = BatchNormalizer(data, batch_size)

for batch in normalizer:
    print(batch)

[[0.0, 0.2, 0.4], [0.6, 0.8, 1.0]]
[[0.0, 0.2, 0.4], [0.6, 0.8, 1.0]]


## Oefening 3:
Moving Average Iterator  

Maak een iterator-klasse genaamd MovingAverage die een numerieke tijdreeks neemt en een voortschrijdend gemiddelde berekent met een opgegeven venstergrootte.  

Vereisten:  

- De iterator retourneert het gemiddelde van elke schuivende vensterpositie.
- Het venster schuift één element per keer op.
- Het venster mag niet groter zijn dan de lengte van de tijdreeks.

In [4]:
# Oplossing

class MovingAverage:
    def __init__(self, data, window_size):
        """
        Initialiseer de iterator met een numerieke tijdreeks en venstergrootte.

        :param data: Een lijst van numerieke waarden
        :param window_size: De grootte van het venster
        """
        if window_size > len(data):
            raise ValueError("Window size mag niet groter zijn dan de lengte van de dataset.")
        self.data = data
        self.window_size = window_size
        self.current_index = 0

    def __iter__(self):
        """Maak het object zelf iterable."""
        return self

    def __next__(self):
        """
        Retourneer het volgende voortschrijdende gemiddelde.

        :raises StopIteration: Als er geen vensters meer zijn
        """
        if self.current_index + self.window_size > len(self.data):
            raise StopIteration

        # Haal het venster op
        window = self.data[self.current_index:self.current_index + self.window_size]

        # Bereken het gemiddelde
        avg = sum(window) / self.window_size

        # Schuif het venster één element op
        self.current_index += 1

        return avg

# Testvoorbeeld
data = [10, 20, 30, 40, 50]
window_size = 3
moving_avg = MovingAverage(data, window_size)

for avg in moving_avg:
    print(avg)

20.0
30.0
40.0


## Oefening 4:  

Genormaliseerde Batch Generator  

Schrijf een generator-functie genaamd batch_normalizer die een dataset (een lijst van lijsten met numerieke waarden) in batches verdeelt en deze batch per batch normaliseert. Normalisatie houdt in dat je van elke waarde in de batch het minimum aftrekt en deelt door het bereik (max - min).    

Vereisten:  

- De generator levert batches van een opgegeven grootte.
- Indien de dataset niet exact kan worden verdeeld, bevat de laatste batch de resterende elementen.
- Normalisatie gebeurt per batch, niet over de hele dataset.


Functiesignatuur:

In [3]:
def batch_normalizer(data, batch_size):
    # Jouw code hier
    pass

In [5]:
def batch_normalizer(data, batch_size):
    """
    Generator-functie die een dataset in batches verdeelt en normaliseert.

    :param data: Een lijst van lijsten met numerieke waarden
    :param batch_size: De grootte van de batches
    :yield: Een genormaliseerde batch
    """
    for i in range(0, len(data), batch_size):
        # Haal de huidige batch
        batch = data[i:i + batch_size]

        # Bepaal minimum, maximum en bereik binnen de batch
        min_val = min(min(row) for row in batch)
        max_val = max(max(row) for row in batch)
        range_val = max_val - min_val

        # Normaliseer de batch
        if range_val == 0:  # Voorkom deling door nul
            normalized_batch = [[0 for _ in row] for row in batch]
        else:
            normalized_batch = [
                [(val - min_val) / range_val for val in row] for row in batch
            ]

        yield normalized_batch

# Testvoorbeeld
data = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
    [10, 11, 12]
]
batch_size = 2
for batch in batch_normalizer(data, batch_size):
    print(batch)

[[0.0, 0.2, 0.4], [0.6, 0.8, 1.0]]
[[0.0, 0.2, 0.4], [0.6, 0.8, 1.0]]


## Oefening 5:  

Moving Average Generator  

Schrijf een generator-functie genaamd moving_average die een numerieke tijdreeks neemt en een voortschrijdend gemiddelde berekent met een opgegeven venstergrootte.  

Vereisten:  

- De generator levert het gemiddelde van elk schuivend venster.
- Het venster schuift één element per keer op.
- Het venster mag niet groter zijn dan de lengte van de tijdreeks.  

    
Functiesignatuur:  

def moving_average(data, window_size):  
        # Jouw code hier


# Testvoorbeeld:
data = [10, 20, 30, 40, 50]  
window_size = 3  
for avg in moving_average(data, window_size):  
    print(avg)

# Output:
20.0  
30.0  
40.0

In [8]:
# oplossing
def moving_average(data, window_size):
    """
    Generator-functie die een voortschrijdend gemiddelde berekent.

    :param data: Een lijst van numerieke waarden
    :param window_size: De grootte van het venster
    :yield: Het gemiddelde van elk schuivend venster
    """
    if window_size > len(data):
        raise ValueError("De venstergrootte mag niet groter zijn dan de lengte van de dataset.")

    for i in range(len(data) - window_size + 1):
        # Selecteer het venster
        window = data[i:i + window_size]

        # Bereken het gemiddelde van het venster
        avg = sum(window) / window_size

        yield avg

# Testvoorbeeld
data = [10, 20, 30, 40, 50]
window_size = 3
for avg in moving_average(data, window_size):
    print(avg)


20.0
30.0
40.0
